# Environment and provenance

Record the runtime and principal dependency versions used to reproduce the demonstrator.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print(f"Python:     {platform.python_version()}")
print(f"NumPy:      {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

# Inputs and configuration

The Python names match the variables listed in the VDT.

In [ ]:
# x: integration upper limits [s]
x_max = np.linspace(0.0, 5.0, 21)

# p: exponential decay rate [s^-1]
exp_decay = 0.8

# Reference implementation

The composite trapezoidal rule evaluates the integral independently for each value of $x_i$.

In [ ]:
def cumulative_response(x_max, exp_decay, quadrature_points=401):
    """Evaluate y_i = p integral_0^x_i exp(-p xi) dxi."""
    if exp_decay <= 0 or np.any(x_max < 0):
        raise ValueError("Require exp_decay > 0 and x_max >= 0")

    resp = np.empty_like(x_max, dtype=float)
    for i, x_i in enumerate(x_max):
        xi = np.linspace(0.0, x_i, quadrature_points)
        resp[i] = exp_decay * np.trapezoid(np.exp(-exp_decay * xi), xi)
    return resp


resp = cumulative_response(x_max, exp_decay)
reference = 1.0 - np.exp(-exp_decay * x_max)
np.testing.assert_allclose(resp, reference, rtol=1e-5, atol=1e-10)
print(f"Maximum absolute error: {np.max(np.abs(resp - reference)):.3e}")

# Results

In [ ]:
figure, axis = plt.subplots(figsize=(6, 3.5), constrained_layout=True)
axis.plot(x_max, reference, label="Closed-form check")
axis.plot(x_max, resp, "o", label="Numerical integral")
axis.set(
    xlabel=r"Input $x$ [s]",
    ylabel=r"Response $y$ [1]",
    title="ALG-001 cumulative response",
)
axis.grid(alpha=0.25)
axis.legend()
plt.show()